In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [ ]:
!pip install wandb

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

api_key = UserSecretsClient().get_secret("WAND")
wandb.login(key=api_key)

In [2]:
import random
import re
import torch
import torch.nn as nn

from torch.optim import Adam
from torch.utils.data import DataLoader
from collections import Counter

In [3]:
random.seed(33)
np.random.seed(33)
torch.manual_seed(33)

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(33)

In [4]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [6]:
OPTIONS = ["A", "B", "C", "D", "E"]

def split_prompt(prompt):
    text = str(prompt).strip()
    prefix, suffix = '', ''

    ci = text.find(': ')
    if ci != -1:
        prefix = text[:ci].strip()
        text = text[ci + 1:].strip()

    n_q = text.count('?')

    if n_q == 1:
        qi = text.index('?')
        suffix = text[qi + 1:].strip()
        text = text[:qi + 1].strip()

    elif n_q == 2:
        first_q = text.index('?')
        prefix = text[:first_q + 1].strip()

        text = text[first_q + 1:].strip()
        second_q = text.index('?')

        suffix = text[second_q + 1:].strip()
        text = text[:second_q + 1].strip()

    return (prefix or None), text, (suffix or None)


def clean(text, is_prompt=True):
    if(is_prompt):
        text = split_prompt(text)[1]
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def tokenize(text, is_prompt=True):
    return clean(text, is_prompt).split()


def ap_at_3(actual, predicted):
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0


def map_at_3(actuals, predictions):
    return float(np.mean([ap_at_3(a, p) for a, p in zip(actuals, predictions)]))


def make_submission(ids, predictions):
    return pd.DataFrame({
        'id': ids,
        'prediction': [' '.join(p[:3]) for p in predictions]
    })

In [7]:
df['q_clean'] = df['prompt'].apply(clean)

for opt in OPTIONS:
    df[f'{opt}_clean'] = df[opt].apply(lambda text: clean(text, is_prompt=False))

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       2000 non-null   int64 
 1   prompt   2000 non-null   object
 2   A        2000 non-null   object
 3   B        2000 non-null   object
 4   C        2000 non-null   object
 5   D        2000 non-null   object
 6   E        2000 non-null   object
 7   answer   2000 non-null   object
 8   q_clean  2000 non-null   object
 9   A_clean  2000 non-null   object
 10  B_clean  2000 non-null   object
 11  C_clean  2000 non-null   object
 12  D_clean  2000 non-null   object
 13  E_clean  2000 non-null   object
dtypes: int64(1), object(13)
memory usage: 218.9+ KB


In [8]:
print("Empty questions:", (df['q_clean'].str.len() == 0).sum())

for opt in OPTIONS:
    empty_count = (df[f'{opt}_clean'].str.len() == 0).sum()
    print(opt, empty_count)

Empty questions: 0
A 0
B 0
C 0
D 0
E 0


In [9]:
unique_qs = df['q_clean'].unique()
(len(df), len(unique_qs), len(set(unique_qs)))

(2000, 252, 252)

In [10]:
NUM_FOLDS = 5

rng = np.random.default_rng(33)
shuffled_qs = unique_qs.copy()
rng.shuffle(shuffled_qs)

fold_qs = np.array_split(shuffled_qs, NUM_FOLDS)

for fold in range(NUM_FOLDS):
    print(f"Fold {fold}: {len(fold_qs[fold])} validation questions")

Fold 0: 51 validation questions
Fold 1: 51 validation questions
Fold 2: 50 validation questions
Fold 3: 50 validation questions
Fold 4: 50 validation questions


In [11]:
def create_pair_dataframe(source_df):
    pair_rows = []

    for row in source_df.itertuples(index=False):
        question_tokens = row.q_clean.split()

        for opt in OPTIONS:
            option_text = getattr(row, f'{opt}_clean')
            option_tokens = option_text.split()

            tokens = question_tokens + ['<SEP>'] + option_tokens

            pair_rows.append({
                'id': row.id,
                'option': opt,
                'tokens': tokens,
                'length': len(tokens),
                'label': int(row.answer == opt)
            })

    return pd.DataFrame(pair_rows)

In [12]:
fold0_val_qs = set(fold_qs[0])
fold0_train_qs = set(unique_qs) - fold0_val_qs

fold0_train_df = df[df['q_clean'].isin(fold0_train_qs)].reset_index(drop=True)
fold0_pairs = create_pair_dataframe(fold0_train_df)

print(fold0_pairs.head(10))
fold0_pairs['length'].describe(percentiles=[0.50, 0.90, 0.95, 0.99])

   id option                                             tokens  length  label
0   1      A  [what, is, martin, heidegger, s, view, on, the...      67      0
1   1      B  [what, is, martin, heidegger, s, view, on, the...      56      1
2   1      C  [what, is, martin, heidegger, s, view, on, the...      53      0
3   1      D  [what, is, martin, heidegger, s, view, on, the...      46      0
4   1      E  [what, is, martin, heidegger, s, view, on, the...      46      0
5   4      A  [what, is, martin, heidegger, s, view, on, the...      67      0
6   4      B  [what, is, martin, heidegger, s, view, on, the...      56      1
7   4      C  [what, is, martin, heidegger, s, view, on, the...      53      0
8   4      D  [what, is, martin, heidegger, s, view, on, the...      46      0
9   4      E  [what, is, martin, heidegger, s, view, on, the...      46      0


count    7730.000000
mean       38.787969
std        19.098676
min        10.000000
50%        34.000000
90%        63.000000
95%        75.000000
99%       100.130000
max       149.000000
Name: length, dtype: float64

In [13]:
class Vocabulary:
    def __init__(self):
        self.t2id = {
            '<PAD>': 0,
            '<UNK>': 1,
            '<SEP>': 2
        }
        self.id2t = {
            0: '<PAD>',
            1: '<UNK>',
            2: '<SEP>'
        }

    def build(self, token_lists):
        counter = Counter()

        for tokens in token_lists:
            counter.update(tokens)

        for token, _ in counter.most_common():
            if token in self.t2id:
                continue
            idx = len(self.t2id)
            self.t2id[token] = idx
            self.id2t[idx] = token

    def encode(self, tokens, max_len):
        tokens = tokens[:max_len]
        ids = [self.t2id.get(token, 1) for token in tokens]

        real_length = len(ids)
        ids += [0] * (max_len - real_length)

        return ids, real_length

    def __len__(self):
        return len(self.t2id)

In [14]:
MAX_LEN = 100

fold0_vocab = Vocabulary()
fold0_vocab.build(fold0_pairs['tokens'])

sample_tokens = fold0_pairs.iloc[0]['tokens']

ids, real_length = fold0_vocab.encode(sample_tokens, max_len=MAX_LEN)

print("Original tokens:", sample_tokens)
print("Encoded IDs:", ids[:real_length])

sep_position = sample_tokens.index('<SEP>')

print("SEP position:", sep_position)
print("SEP ID:", ids[sep_position])

decoded_tokens = [fold0_vocab.id2t[idx] for idx in ids[:real_length]]
decoded_tokens

Original tokens: ['what', 'is', 'martin', 'heidegger', 's', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', '<SEP>', 'martin', 'heidegger', 'believes', 'that', 'humans', 'exist', 'within', 'a', 'time', 'continuum', 'that', 'is', 'infinite', 'and', 'does', 'not', 'have', 'a', 'defined', 'beginning', 'or', 'end', 'the', 'relationship', 'to', 'the', 'past', 'involves', 'acknowledging', 'it', 'as', 'a', 'historical', 'era', 'and', 'the', 'relationship', 'to', 'the', 'future', 'involves', 'creating', 'a', 'world', 'that', 'will', 'endure', 'beyond', 'one', 's', 'own', 'time']
Encoded IDs: [8, 5, 433, 434, 17, 435, 23, 3, 40, 24, 30, 9, 512, 187, 2, 433, 434, 1139, 11, 1346, 423, 163, 6, 30, 1686, 11, 5, 782, 9, 99, 43, 55, 6, 210, 1506, 16, 1990, 3, 40, 10, 3, 798, 585, 1991, 22, 18, 6, 1992, 1993, 9, 3, 40, 10, 3, 949, 585, 412, 6, 1361, 11, 112, 1994, 1995, 57, 17, 1592, 30]
SEP position: 14
SEP ID: 2


['what',
 'is',
 'martin',
 'heidegger',
 's',
 'view',
 'on',
 'the',
 'relationship',
 'between',
 'time',
 'and',
 'human',
 'existence',
 '<SEP>',
 'martin',
 'heidegger',
 'believes',
 'that',
 'humans',
 'exist',
 'within',
 'a',
 'time',
 'continuum',
 'that',
 'is',
 'infinite',
 'and',
 'does',
 'not',
 'have',
 'a',
 'defined',
 'beginning',
 'or',
 'end',
 'the',
 'relationship',
 'to',
 'the',
 'past',
 'involves',
 'acknowledging',
 'it',
 'as',
 'a',
 'historical',
 'era',
 'and',
 'the',
 'relationship',
 'to',
 'the',
 'future',
 'involves',
 'creating',
 'a',
 'world',
 'that',
 'will',
 'endure',
 'beyond',
 'one',
 's',
 'own',
 'time']

In [15]:
def unk_stats(pair_df, vocab):
    total, unknown, pairs_with_unk = 0, 0, 0
    unk_counter = Counter()

    for tokens in pair_df['tokens']:
        has_unk = False
        for token in tokens:
            total += 1
            if token not in vocab.t2id:
                unknown += 1
                unk_counter[token] += 1
                has_unk = True
        pairs_with_unk += int(has_unk)

    return total, unknown, unk_counter, pairs_with_unk

fold0_val_df = df[df['q_clean'].isin(fold0_val_qs)].reset_index(drop=True)
fold0_val_pairs = create_pair_dataframe(fold0_val_df)

total_tokens, unknown_tokens, unknown_counter, pairs_with_unk = unk_stats(fold0_val_pairs, fold0_vocab)

print("Validation UNK rate:", unknown_tokens / total_tokens)
print("Validation pair-level UNK %:", pairs_with_unk / len(fold0_val_pairs) * 100)
print("Unique unknown tokens:", len(unknown_counter))
print("Most common unknown tokens:", unknown_counter.most_common(20))

Validation UNK rate: 0.1671046046046046
Validation pair-level UNK %: 94.09691629955947
Unique unknown tokens: 400
Most common unknown tokens: [('fatty', 229), ('l', 220), ('acyl', 220), ('thylakoid', 220), ('membranes', 220), ('fusion', 209), ('disorder', 200), ('hysteresis', 190), ('b', 180), ('environmental', 150), ('alloys', 150), ('propagation', 150), ('memristors', 143), ('simultaneity', 140), ('isaac', 140), ('pinched', 140), ('curves', 140), ('reram', 140), ('terminal', 140), ('dragging', 135)]


In [16]:
def encode_groups(pair_df, vocab, max_len, has_labels=True):
    samples = []

    for question_id, group in pair_df.groupby('id', sort=False):
        group = group.sort_values('option')

        input_ids, lengths = [], []

        for row in group.itertuples(index=False):
            row_ids, row_length = vocab.encode(row.tokens, max_len)
            input_ids.append(row_ids)
            lengths.append(row_length)

        sample = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'length': torch.tensor(lengths, dtype=torch.long),
            'id': question_id
        }

        if has_labels:
            label_idx = group['label'].tolist().index(1)
            sample['label'] = torch.tensor(label_idx, dtype=torch.long)

        samples.append(sample)

    return samples

In [17]:
BATCH_SIZE = 64

In [18]:
class BiLSTMMCQ(nn.Module):
    def __init__(self, vocab_size, padding_idx, embed_dim=128, hidden_dim=128, num_layers=1, dropout=0.3):
        super().__init__()

        self.padding_idx = padding_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx)
        self.embedding_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, input_ids, lengths):
        batch_size, num_options, seq_len = input_ids.shape

        input_ids = input_ids.view(batch_size * num_options, seq_len)
        lengths = lengths.view(batch_size * num_options)

        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=seq_len
        )

        mask = (input_ids != self.padding_idx).unsqueeze(-1).to(output.dtype)
        pooled_output = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        logits = self.classifier(pooled_output).squeeze(-1)
        logits = logits.view(batch_size, num_options)

        return logits

In [21]:
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()

In [22]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_samples = 0.0, 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        lengths = batch['length']
        labels = batch['label'].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

In [23]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    predictions, actual_answers = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        lengths = batch['length']
        labels = batch['label'].to(device)

        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        labels_cpu = labels.cpu().numpy()

        for probability_row, label in zip(probabilities, labels_cpu):
            ranked_options = [OPTIONS[idx] for idx in np.argsort(-probability_row)]
            predictions.append(ranked_options[:3])
            actual_answers.append(OPTIONS[label])

    return {
        'loss': total_loss / total_samples,
        'map3': map_at_3(actual_answers, predictions),
        'predictions': predictions,
        'actual_answers': actual_answers
    }

In [ ]:
EPOCHS = 150
PATIENCE = 30

def run_fold(fold, fold_qs):
    val_qs = set(fold_qs[fold])
    train_qs = set(unique_qs) - val_qs

    train_df = df[df['q_clean'].isin(train_qs)].reset_index(drop=True)
    val_df = df[df['q_clean'].isin(val_qs)].reset_index(drop=True)

    train_pairs = create_pair_dataframe(train_df)
    val_pairs = create_pair_dataframe(val_df)

    vocab = Vocabulary()
    vocab.build(train_pairs['tokens'])

    train_samples = encode_groups(train_pairs, vocab, MAX_LEN, has_labels=True)
    val_samples = encode_groups(val_pairs, vocab, MAX_LEN, has_labels=True)

    train_loader = DataLoader(train_samples, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_samples, batch_size=BATCH_SIZE*2, shuffle=False)

    model = BiLSTMMCQ(
        vocab_size=len(vocab),
        padding_idx=0,
        embed_dim=128,
        hidden_dim=128,
        num_layers=1,
        dropout=0.3
    ).to(device)

    optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

    best_map3 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    history = []
    checkpoint_path = f'best_bilstm_fold{fold}.pt'

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        validation_result = evaluate(model, val_loader, criterion, device)

        val_loss = validation_result['loss']
        val_map3 = validation_result['map3']

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_map3': val_map3
        })

        print(f"Fold {fold} | Epoch {epoch} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val MAP@3: {val_map3:.4f}")

        run.log({
            "fold": fold,
            "epoch": epoch,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "val/map3": val_map3
        })

        if val_map3 > best_map3:
            best_map3 = val_map3
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(model.state_dict(), checkpoint_path)
            print(f"Fold {fold}: Best model saved: MAP@3={best_map3:.4f}")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"Early stopping after epoch {epoch}.")
                break

    print(f"\nFold {fold} training complete")
    print(f"Fold {fold}: Best epoch: {best_epoch}")
    print(f"Fold {fold}: Best validation MAP@3: {best_map3}\n")

    run.log({
        "fold": fold,
        "fold/best_epoch": best_epoch,
        "fold/best_map3": best_map3
    })

    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.eval()

    best_validation_result = evaluate(model, val_loader, criterion, device)

    oof_ids = val_pairs['id'].unique().tolist()

    return {
        'fold': fold,
        'history': pd.DataFrame(history),
        'best_map3': best_map3,
        'model': model,
        'vocab': vocab,
        'oof_ids': oof_ids,
        'oof_predictions': best_validation_result['predictions'],
        'oof_actuals': best_validation_result['actual_answers']
    }

In [ ]:
run = wandb.init(
    project="23f2003391-t22026",
    name="bilstm-kfolds",
    config={
        "model": 'bilstm-scratch',
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "folds": NUM_FOLDS,
        "max_length": MAX_LEN,
    },
)

In [25]:
fold_results = []

for fold in range(NUM_FOLDS):
    fold_results.append(run_fold(fold, fold_qs))

Fold 0 | Epoch 1 | Train loss: 1.5049 | Val loss: 1.5527 | Val MAP@3: 0.4633
Fold 0: Best model saved: MAP@3=0.4633
Fold 0 | Epoch 2 | Train loss: 1.1924 | Val loss: 1.5937 | Val MAP@3: 0.4633
Fold 0 | Epoch 3 | Train loss: 0.9137 | Val loss: 1.6279 | Val MAP@3: 0.4457
Fold 0 | Epoch 4 | Train loss: 0.6785 | Val loss: 1.7828 | Val MAP@3: 0.4618
Fold 0 | Epoch 5 | Train loss: 0.4810 | Val loss: 1.9993 | Val MAP@3: 0.4369
Fold 0 | Epoch 6 | Train loss: 0.3875 | Val loss: 1.9868 | Val MAP@3: 0.4974
Fold 0: Best model saved: MAP@3=0.4974
Fold 0 | Epoch 7 | Train loss: 0.3167 | Val loss: 1.7994 | Val MAP@3: 0.4519
Fold 0 | Epoch 8 | Train loss: 0.2277 | Val loss: 2.3530 | Val MAP@3: 0.4325
Fold 0 | Epoch 9 | Train loss: 0.2301 | Val loss: 1.9890 | Val MAP@3: 0.4879
Fold 0 | Epoch 10 | Train loss: 0.1635 | Val loss: 1.9942 | Val MAP@3: 0.4593
Fold 0 | Epoch 11 | Train loss: 0.1510 | Val loss: 1.9318 | Val MAP@3: 0.5103
Fold 0: Best model saved: MAP@3=0.5103
Fold 0 | Epoch 12 | Train loss: 0.

In [26]:
oof_predictions, oof_actuals = [], []

for result in fold_results:
    oof_predictions.extend(result['oof_predictions'])
    oof_actuals.extend(result['oof_actuals'])
    print(f"Fold {result['fold']} best validation MAP@3: {result['best_map3']:.4f}")

overall_map3 = map_at_3(oof_actuals, oof_predictions)
print("Overall OOF MAP@3:", overall_map3)

Fold 0 best validation MAP@3: 0.5253
Fold 1 best validation MAP@3: 0.5453
Fold 2 best validation MAP@3: 0.5643
Fold 3 best validation MAP@3: 0.6360
Fold 4 best validation MAP@3: 0.6375
Overall OOF MAP@3: 0.5774166666666666


In [ ]:
run.summary["oof/map3"] = overall_map3
run.finish()

In [27]:
history_df = pd.concat(
    [result['history'].assign(fold=result['fold']) for result in fold_results],
    ignore_index=True
)
history_df

,epoch,train_loss,val_loss,val_map3,fold
0,1,1.504937,1.552719,0.463289,0
1,2,1.192424,1.593742,0.463289,0
2,3,0.913744,1.627862,0.445668,0
3,4,0.678537,1.782835,0.461821,0
4,5,0.481041,1.999312,0.436858,0
...,...,...,...,...,...
298,42,0.004343,2.044768,0.577213,4
299,43,0.005406,2.084073,0.572505,4
300,44,0.014371,2.526322,0.576271,4
301,45,0.008828,2.578330,0.530132,4


In [28]:
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
test_df['q_clean'] = test_df['prompt'].apply(clean)

for opt in OPTIONS:
    test_df[f'{opt}_clean'] = test_df[opt].apply(lambda text: clean(text, is_prompt=False))

print(test_df.shape)
test_df.head()

(500, 13)


,id,prompt,A,B,C,D,E,q_clean,A_clean,B_clean,C_clean,D_clean,E_clean
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",what is the relationship between the hamiltoni...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",what is the estimated redshift of ceers 93316 ...,approximately z 6 0 corresponding to 1 billion...,approximately z 16 7 corresponding to 235 8 mi...,approximately z 3 0 corresponding to 5 billion...,approximately z 10 0 corresponding to 13 billi...,approximately z 13 0 corresponding to 30 billi...
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...,what is the reason for the sun appearing sligh...,the sun appears yellowish due to a reflection ...,the longer wavelengths of light such as red an...,the sun appears yellowish due to the scatterin...,the sun emits a yellow light due to its own sp...,the atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,what is the significance of the redshift dista...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,what is the landau lifshitz gilbert equation u...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...


In [29]:
def create_test_pair_dataframe(source_df):
    pair_rows = []

    for row in source_df.itertuples(index=False):
        question_tokens = row.q_clean.split()

        for opt in OPTIONS:
            option_text = getattr(row, f'{opt}_clean')
            option_tokens = option_text.split()
            tokens = question_tokens + ['<SEP>'] + option_tokens

            pair_rows.append({
                'id': int(row.id),
                'option': opt,
                'tokens': tokens,
                'length': len(tokens)
            })

    return pd.DataFrame(pair_rows)

In [30]:
test_pairs = create_test_pair_dataframe(test_df)

(len(test_df), len(test_pairs))

(500, 2500)

In [31]:
@torch.no_grad()
def predict_test(model, loader, device):
    model.eval()

    ids_list, probabilities_list = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        lengths = batch['length']

        logits = model(input_ids, lengths)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

        ids_cpu = batch['id'].cpu().numpy()

        ids_list.extend(ids_cpu.tolist())
        probabilities_list.extend(probabilities)

    return ids_list, probabilities_list

In [32]:
fold_probabilities = []

for result in fold_results:
    test_samples = encode_groups(test_pairs, result['vocab'], MAX_LEN, has_labels=False)
    test_loader = DataLoader(test_samples, batch_size=BATCH_SIZE*2, shuffle=False)

    test_ids, test_probs = predict_test(result['model'], test_loader, device)

    fold_probabilities.append(np.array(test_probs))

ensemble_probabilities = np.mean(fold_probabilities, axis=0)

In [33]:
test_predictions = [
    [OPTIONS[idx] for idx in np.argsort(-probability_row)][:3]
    for probability_row in ensemble_probabilities
]

submission = make_submission(test_ids, test_predictions)
submission.to_csv('submission.csv', index=False)

submission.head(5)

,id,prediction
0,1,A C D
1,2,B A C
2,3,B D E
3,4,E A C
4,5,C E A
